In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from sometria.catalog import (
    MotionViewSpec,
    build_motion_view,
    load_annotations,
    load_catalog,
    load_label_vocabulary,
    load_splits,
)
from sometria.preprocess import ImportConfig, import_opensim_csv_dataset
from sometria.carepd import (
    import_carepd_annotations,
    import_carepd_gait_vocabulary,
    import_carepd_subject_splits,
)
from sometria.representation import Representation, _load_human_definition

# CARE-PD

CARE-PD reaches us as the same OpenSim CSV as AMASS and MotionX, so the motion side is
the same import call. Two things differ, and both are why this notebook exists rather
than a third cell in `preprocess_motionx`:

- **The export is flat.** `<subset>__<subject>__<take>.csv` with no directories, so the
  subset comes from the filename prefix rather than the leading folder. `_source_subset`
  in `sometria.preprocess` handles both layouts.
- **It is an evaluation corpus, not pretraining data.** MoCHA labels 871 of the takes
  with an MDS-UPDRS gait score, and the probe splits by subject. Nothing here calls
  `create_pretrain_split`: training on CARE-PD would train on the probe's own patients.

In [ ]:
RAW_CAREPD = Path("/home/fziche/nas/MAEVE/HUMAN_MODEL/CARE-PD_torque")
RAW_MOCHA = Path("/home/fziche/nas/MAEVE/HUMAN_MODEL/CARE_PD/mocha_training_dataset")
OUT = Path("../data/processed")

HUMAN = _load_human_definition("../config/human.yaml")
REPRESENTATION = Representation.from_human(HUMAN)
print(REPRESENTATION)

## Import

`pattern="*.csv"` rather than `"**/*.csv"` on purpose. The export root also holds an
older nested copy of itself under `CARE-PD_torque/`, plus ~220 empty directories left by
takes whose conversion failed. A recursive glob would import the stale copy a second time
under different sample ids.

In [ ]:
catalog = import_opensim_csv_dataset(
    config=ImportConfig(
        source_dataset="CARE-PD",
        input_root=RAW_CAREPD,
        output_root=OUT,
        pattern="*.csv",
    ),
    representation=REPRESENTATION,
    human=HUMAN,
)

print(f"{len(catalog):,} catalog rows total")
catalog.head()

## Labels and splits

Three separate tables for the same reason BABEL uses three: a sample can be imported
without being labelled (7,588 of the takes are), and a benchmark's admissible answers are
a different question from what any one sample is.

In [ ]:
annotations = import_carepd_annotations(output_root=OUT, mocha_root=RAW_MOCHA)
vocabulary = import_carepd_gait_vocabulary(output_root=OUT)
splits = import_carepd_subject_splits(output_root=OUT, mocha_root=RAW_MOCHA)

print(f"{len(annotations):,} annotation rows | {len(vocabulary)} gait classes")

## Sanity checks

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

catalog = load_catalog(OUT)
splits = load_splits(OUT)
annotations = load_annotations(OUT)

carepd = catalog.filter(pl.col("source_dataset") == "CARE-PD")
labels = annotations.filter(pl.col("label_source") == "CARE-PD")

print(f"{len(carepd):,} CARE-PD samples across {carepd['source_subset'].n_unique()} subsets")
print(f"{carepd['duration'].sum() / 3600:.1f} motion hours ({carepd['duration'].median():.1f}s median take)")
print(f"{(~carepd['broken']).sum():,} clean | {carepd['broken'].sum():,} broken")
print(f"{labels['sample_id'].n_unique():,} labelled samples")

carepd.head(3)

### Subsets and takes

CARE-PD takes are short -- a few seconds of walking, not a minutes-long mocap session --
so the `min_frames` a pretraining view uses would discard most of the corpus. A probe
window has to be sized against this distribution, not against AMASS's.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))

subset_summary = (
    carepd
    .group_by("source_subset")
    .agg(
        pl.len().alias("samples"),
        (~pl.col("broken")).sum().alias("clean"),
        (pl.col("duration").sum() / 3600).alias("hours"),
    )
    .sort("samples")
)

y = np.arange(len(subset_summary))
ax[0].barh(y - 0.2, subset_summary["samples"], height=0.4, color="#999999", label="all")
ax[0].barh(y + 0.2, subset_summary["clean"], height=0.4, color="#4878a8", label="clean")
ax[0].set_yticks(y, subset_summary["source_subset"])
ax[0].set(title="Samples per subset", xlabel="samples")
ax[0].legend()

ax[1].hist(carepd["duration"].to_numpy(), bins=60, color="#4878a8")
ax[1].axvline(240 / 60, color="#d1615d", linewidth=1.5, label="240 frames @ 60 Hz")
ax[1].set(title="Take duration", xlabel="seconds", ylabel="samples")
ax[1].legend()

ax[2].hist(carepd["original_hz"].to_numpy(), bins=60, color="#6acc64")
ax[2].set(title="Capture rate before resampling", xlabel="Hz", ylabel="samples")

fig.tight_layout()

print(subset_summary.sort("samples", descending=True))

### Torque quality

Same check as the other corpora, on the raw OpenSim channels before `Representation.encode`,
so `tau_rate` stays in physical units.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

clean = carepd.filter(~pl.col("broken"))
broken = carepd.filter(pl.col("broken"))

ax[0].hist(np.log10(clean["tau_rate"].to_numpy().clip(1)), bins=60, alpha=0.8, color="#4878a8", label="clean")
if len(broken):
    ax[0].hist(np.log10(broken["tau_rate"].to_numpy().clip(1)), bins=60, alpha=0.8, color="#d1615d", label="broken")
ax[0].set(xlabel="log10 max |dtau/dt|", ylabel="samples", title="Torque discontinuity")
ax[0].legend()

bad_by_subset = (
    carepd
    .group_by("source_subset")
    .agg(pl.len().alias("n"), pl.col("broken").sum().alias("bad"))
    .with_columns((100 * pl.col("bad") / pl.col("n")).alias("bad_pct"))
    .sort("bad_pct")
)

ax[1].barh(bad_by_subset["source_subset"], bad_by_subset["bad_pct"], color="#d1615d")
ax[1].set(xlabel="% broken", title="Broken rate by subset")

fig.tight_layout()

### Label balance and split disjointness

Two things have to hold before the probe means anything: the gait scores are imbalanced
(so the probe reports balanced accuracy, not accuracy), and no subject appears in two
splits (so it scores gait rather than patient identity).

In [ ]:
carepd_split = splits.filter(pl.col("split_set") == "carepd_subject")

labelled = (
    labels
    .join(carepd_split.select("sample_id", "split"), on="sample_id", how="inner")
    .join(carepd.select("sample_id", "source_subset", "source_path"), on="sample_id", how="inner")
    .with_columns(
        pl.col("source_path").str.split("__").list.slice(0, 2).list.join("__").alias("subject")
    )
)

subject_splits = labelled.group_by("subject").agg(pl.col("split").n_unique().alias("splits"))
leaked = subject_splits.filter(pl.col("splits") > 1)
assert leaked.is_empty(), f"{len(leaked)} subjects appear in more than one split"
print(f"OK: {len(subject_splits)} subjects, each in exactly one split")

balance = (
    labelled
    .group_by("split", "label")
    .len()
    .pivot(on="split", index="label", values="len")
    .sort("label")
    .fill_null(0)
)
print(balance)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

split_order = ["train", "val", "test"]
width = 0.8 / len(split_order)
x = np.arange(len(balance))
for i, split in enumerate(split_order):
    if split in balance.columns:
        ax[0].bar(x + (i - 1) * width, balance[split], width=width, label=split)
ax[0].set_xticks(x, balance["label"])
ax[0].set(title="Gait score per split", xlabel="MDS-UPDRS 3.10 score", ylabel="samples")
ax[0].legend()

subject_counts = (
    labelled.group_by("split").agg(pl.col("subject").n_unique().alias("subjects"), pl.len().alias("samples"))
)
sx = np.arange(len(subject_counts))
ax[1].bar(sx - 0.2, subject_counts["subjects"], width=0.4, color="#4878a8", label="subjects")
ax[1].bar(sx + 0.2, subject_counts["samples"], width=0.4, color="#999999", label="samples")
ax[1].set_xticks(sx, subject_counts["split"])
ax[1].set(title="Split sizes", ylabel="count")
ax[1].legend()

fig.tight_layout()

### CARE-PD stays out of pretraining

`create_pretrain_split` defaults to "every corpus in the catalog", which was correct while
every corpus was pretraining data. It is not correct now. Re-running the AMASS or MotionX
notebook must pass `source_datasets=("AMASS", "MotionX")`, or the probe's own patients end
up in the pretraining view. This cell fails loudly if that has happened.

In [ ]:
pretrain_ids = splits.filter(pl.col("split_set").str.starts_with("pretrain"))

leaked = (
    carepd
    .join(pretrain_ids.select("sample_id", "split_set").unique(), on="sample_id", how="inner")
)
assert leaked.is_empty(), (
    f"{len(leaked)} CARE-PD samples are in {leaked['split_set'].unique().to_list()}; "
    "rebuild those splits with source_datasets=(\"AMASS\", \"MotionX\")"
)
print("OK: no CARE-PD sample is in any pretraining split")

### What a probe view sees

In [ ]:
rows = []
for split in ("train", "val", "test"):
    view = build_motion_view(
        OUT,
        MotionViewSpec(
            split_set="carepd_subject",
            split=split,
            source_datasets=("CARE-PD",),
            label_sources=("CARE-PD",),
            require_labels=True,
            exclude_broken=True,
        ),
    )
    rows.append({
        "split": split,
        "samples": len(view),
        "hours": view["duration"].sum() / 3600,
        "median_s": view["duration"].median(),
        "min_frames": view["n_frames"].min(),
    })

print(pl.DataFrame(rows))
print(load_label_vocabulary(OUT, "carepd_updrs_gait"))